In [1]:
# Customer Risk Intelligence
# 03 - Probability Calibration

print("Probability calibration analysis")

Probability calibration analysis


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay
)
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    log_loss
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
DATA_PATH = Path("../data/raw/telco_customer_churn.csv")

df = pd.read_csv(DATA_PATH)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Dataset shape:", df.shape)

Dataset shape: (7043, 21)


# Research Feature Engineering

The research model extends the baseline feature set with business-oriented engineered variables designed to represent customer tenure and revenue characteristics.

The engineered features are:

* **AnnualRevenue** — estimated annualized recurring revenue based on MonthlyCharges.
* **TenureGroup** — categorical representation of customer tenure.
* **MonthlyChargeGroup** — categorical representation of monthly charge level.

These features are introduced to evaluate whether business-oriented feature engineering improves churn prediction and downstream customer-risk analysis.



In [4]:
# Research feature engineering

df_research = df.copy()

# Annualized recurring revenue
df_research["AnnualRevenue"] = (
    df_research["MonthlyCharges"] * 12
)

# Tenure groups
df_research["TenureGroup"] = pd.cut(
    df_research["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

# Monthly charge groups
df_research["MonthlyChargeGroup"] = pd.cut(
    df_research["MonthlyCharges"],
    bins=[-np.inf, 40, 70, 90, np.inf],
    labels=[
        "Low",
        "Medium-Low",
        "Medium-High",
        "High"
    ]
)

print("Research features created successfully.")
print(
    df_research[
        [
            "MonthlyCharges",
            "AnnualRevenue",
            "tenure",
            "TenureGroup",
            "MonthlyChargeGroup"
        ]
    ].head()
)

Research features created successfully.
   MonthlyCharges  AnnualRevenue  tenure   TenureGroup MonthlyChargeGroup
0           29.85          358.2       1    0-6 months                Low
1           56.95          683.4      34  25-48 months         Medium-Low
2           53.85          646.2       2    0-6 months         Medium-Low
3           42.30          507.6      45  25-48 months         Medium-Low
4           70.70          848.4       2    0-6 months        Medium-High


In [5]:
print("Research dataset shape:", df_research.shape)

print("\nNew research features:")
print(
    [
        "AnnualRevenue",
        "TenureGroup",
        "MonthlyChargeGroup"
    ]
)

Research dataset shape: (7043, 24)

New research features:
['AnnualRevenue', 'TenureGroup', 'MonthlyChargeGroup']


## Research Random Forest Model

The research model extends the baseline Random Forest by incorporating business-oriented engineered features.

A stratified train-test split is used to preserve the original churn distribution in both subsets. Numerical and categorical variables are processed through a unified preprocessing pipeline, ensuring that transformations are learned only from the training data.

The resulting model will provide the probability estimates that are subsequently evaluated and calibrated for use in customer risk and retention analysis.


In [6]:
# Research model dataset

target = "Churn"

X_research = df_research.drop(
    columns=["customerID", target]
)

y_research = df_research[target].astype(int)

print("Research feature matrix shape:", X_research.shape)
print("Research target shape:", y_research.shape)
print("\nTarget distribution:")
print(y_research.value_counts())

Research feature matrix shape: (7043, 22)
Research target shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
# Stratified train-test split for research model

X_train_research, X_test_research, y_train_research, y_test_research = train_test_split(
    X_research,
    y_research,
    test_size=0.20,
    stratify=y_research,
    random_state=42
)

print("Training samples:", X_train_research.shape[0])
print("Testing samples:", X_test_research.shape[0])

print("\nTraining churn rate:")
print(y_train_research.mean())

print("\nTesting churn rate:")
print(y_test_research.mean())

Training samples: 5634
Testing samples: 1409

Training churn rate:
0.2653532126375577

Testing churn rate:
0.2654364797728886


In [8]:
# Identify numerical and categorical features

numeric_features_research = X_research.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_research = X_research.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical features:")
print(numeric_features_research)

print("\nCategorical features:")
print(categorical_features_research)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AnnualRevenue']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup', 'MonthlyChargeGroup']


In [9]:
# Research preprocessing pipeline

numeric_transformer_research = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer_research = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor_research = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer_research,
            numeric_features_research
        ),
        (
            "cat",
            categorical_transformer_research,
            categorical_features_research
        )
    ]
)

print("Research preprocessing pipeline created successfully.")

Research preprocessing pipeline created successfully.


In [10]:
# Research Random Forest pipeline

research_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_research
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            )
        )
    ]
)

print("Research Random Forest pipeline created successfully.")

Research Random Forest pipeline created successfully.


In [11]:
# Train Research Random Forest

research_rf_pipeline.fit(
    X_train_research,
    y_train_research
)

print("Research Random Forest trained successfully.")

Research Random Forest trained successfully.


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [13]:
# Research Random Forest evaluation

y_pred_research = research_rf_pipeline.predict(
    X_test_research
)

y_prob_research = research_rf_pipeline.predict_proba(
    X_test_research
)[:, 1]

research_accuracy = accuracy_score(
    y_test_research,
    y_pred_research
)

research_precision = precision_score(
    y_test_research,
    y_pred_research
)

research_recall = recall_score(
    y_test_research,
    y_pred_research
)

research_f1 = f1_score(
    y_test_research,
    y_pred_research
)

research_roc_auc = roc_auc_score(
    y_test_research,
    y_prob_research
)

research_pr_auc = average_precision_score(
    y_test_research,
    y_prob_research
)

print("=" * 60)
print("RESEARCH RANDOM FOREST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:  {research_accuracy:.4f}")
print(f"Precision: {research_precision:.4f}")
print(f"Recall:    {research_recall:.4f}")
print(f"F1 Score:  {research_f1:.4f}")
print(f"ROC-AUC:   {research_roc_auc:.4f}")
print(f"PR-AUC:    {research_pr_auc:.4f}")

RESEARCH RANDOM FOREST PERFORMANCE
Accuracy:  0.7686
Precision: 0.5548
Recall:    0.6497
F1 Score:  0.5985
ROC-AUC:   0.8208
PR-AUC:    0.6069


In [14]:
# Store research-model probability outputs with customer IDs

research_predictions = X_test_research.copy()

research_predictions["customerID"] = df.loc[
    X_test_research.index,
    "customerID"
].values

research_predictions["ActualChurn"] = y_test_research.values
research_predictions["ChurnProbability"] = y_prob_research
research_predictions["PredictedChurn"] = y_pred_research

print("Research prediction dataset created successfully.")
print("Shape:", research_predictions.shape)

research_predictions.head()

Research prediction dataset created successfully.
Shape: (1409, 26)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,AnnualRevenue,TenureGroup,MonthlyChargeGroup,customerID,ActualChurn,ChurnProbability,PredictedChurn
437,Male,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),114.05,8468.20,1368.6,49-72 months,High,4376-KFVRS,0,0.000000,0
2280,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.15,908.55,1201.8,7-12 months,High,2754-SDJRD,0,0.820000,1
2235,Female,0,Yes,Yes,41,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,No,One year,Yes,Credit card (automatic),78.35,3211.20,940.2,25-48 months,Medium-High,9917-KWRBE,0,0.083333,0
4460,Male,0,Yes,No,18,Yes,No,Fiber optic,No,No,Yes,Yes,No,No,Month-to-month,No,Electronic check,78.20,1468.75,938.4,13-24 months,Medium-High,0365-GXEZS,0,0.380000,0
3761,Female,0,Yes,No,72,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),82.65,5919.35,991.8,49-72 months,Medium-High,9385-NXKDA,0,0.030000,0


In [15]:
# Evaluate probability quality of the research model

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

research_brier_original = brier_score_loss(
    y_test_research,
    y_prob_research
)

research_logloss_original = log_loss(
    y_test_research,
    y_prob_research
)

research_roc_auc_original = roc_auc_score(
    y_test_research,
    y_prob_research
)

print("=" * 60)
print("RESEARCH MODEL CALIBRATION METRICS")
print("=" * 60)

print(f"Brier Score: {research_brier_original:.4f}")
print(f"ROC-AUC:     {research_roc_auc_original:.4f}")
print(f"Log Loss:    {research_logloss_original:.4f}")

RESEARCH MODEL CALIBRATION METRICS
Brier Score: 0.1562
ROC-AUC:     0.8208
Log Loss:    0.4768


In [16]:
# Probability calibration for the research Random Forest
# Calibration is performed using cross-validation on the training data.
# The held-out test set is reserved for final evaluation.

from sklearn.calibration import CalibratedClassifierCV

calibrated_research_rf = CalibratedClassifierCV(
    estimator=research_rf_pipeline,
    method="sigmoid",
    cv=5
)

calibrated_research_rf.fit(
    X_train_research,
    y_train_research
)

y_prob_calibrated_research = calibrated_research_rf.predict_proba(
    X_test_research
)[:, 1]

print("Calibrated research-model probabilities generated.")

print(
    pd.Series(
        y_prob_calibrated_research,
        name="CalibratedChurnProbability"
    ).describe()
)

Calibrated research-model probabilities generated.
count    1409.000000
mean        0.268284
std         0.230420
min         0.052984
25%         0.075386
50%         0.167457
75%         0.421586
max         0.831003
Name: CalibratedChurnProbability, dtype: float64


In [17]:
# Evaluate calibrated probability quality on the held-out test set

research_brier_calibrated = brier_score_loss(
    y_test_research,
    y_prob_calibrated_research
)

research_logloss_calibrated = log_loss(
    y_test_research,
    y_prob_calibrated_research
)

research_roc_auc_calibrated = roc_auc_score(
    y_test_research,
    y_prob_calibrated_research
)

print("=" * 60)
print("RESEARCH MODEL CALIBRATION COMPARISON")
print("=" * 60)

print(f"{'Metric':<15}{'Original':>12}{'Calibrated':>15}")
print("-" * 42)

print(
    f"{'Brier Score':<15}"
    f"{research_brier_original:>12.4f}"
    f"{research_brier_calibrated:>15.4f}"
)

print(
    f"{'ROC-AUC':<15}"
    f"{research_roc_auc_original:>12.4f}"
    f"{research_roc_auc_calibrated:>15.4f}"
)

print(
    f"{'Log Loss':<15}"
    f"{research_logloss_original:>12.4f}"
    f"{research_logloss_calibrated:>15.4f}"
)

RESEARCH MODEL CALIBRATION COMPARISON
Metric             Original     Calibrated
------------------------------------------
Brier Score          0.1562         0.1431
ROC-AUC              0.8208         0.8276
Log Loss             0.4768         0.4397


In [18]:
# Quantify calibration improvement

brier_improvement = (
    research_brier_original -
    research_brier_calibrated
)

logloss_improvement = (
    research_logloss_original -
    research_logloss_calibrated
)

roc_auc_change = (
    research_roc_auc_calibrated -
    research_roc_auc_original
)

print("=" * 60)
print("CALIBRATION IMPROVEMENT")
print("=" * 60)

print(f"Brier score improvement: {brier_improvement:.4f}")
print(f"Log loss improvement:    {logloss_improvement:.4f}")
print(f"ROC-AUC change:          {roc_auc_change:.4f}")

CALIBRATION IMPROVEMENT
Brier score improvement: 0.0131
Log loss improvement:    0.0371
ROC-AUC change:          0.0068


In [19]:
# Add calibrated probabilities to the research prediction dataset

research_predictions["CalibratedChurnProbability"] = (
    y_prob_calibrated_research
)

print("Calibrated probabilities added successfully.")

research_predictions[
    [
        "customerID",
        "ActualChurn",
        "ChurnProbability",
        "CalibratedChurnProbability",
        "PredictedChurn"
    ]
].head(10)

Calibrated probabilities added successfully.


,customerID,ActualChurn,ChurnProbability,CalibratedChurnProbability,PredictedChurn
437,4376-KFVRS,0,0.000000,0.054746,0
2280,2754-SDJRD,0,0.820000,0.704098,1
2235,9917-KWRBE,0,0.083333,0.085891,0
4460,0365-GXEZS,0,0.380000,0.272638,0
3761,9385-NXKDA,0,0.030000,0.058594,0
5748,4686-UXDML,0,0.533333,0.468956,1
3568,2227-JRSJX,0,0.476667,0.392529,0
2976,4830-FAXFM,0,0.163333,0.107743,0
5928,1830-IPXVJ,0,0.023333,0.056961,0
1639,4690-LLKUA,1,0.470000,0.392435,0


In [20]:
# Assign customer risk levels using calibrated churn probability

research_predictions["CalibratedRiskLevel"] = pd.cut(
    research_predictions["CalibratedChurnProbability"],
    bins=[-float("inf"), 0.30, 0.60, float("inf")],
    labels=["Low", "Medium", "High"],
    right=False
)

print("Risk levels assigned successfully.")

print(
    research_predictions["CalibratedRiskLevel"]
    .value_counts()
    .sort_index()
)

Risk levels assigned successfully.
CalibratedRiskLevel
Low       917
Medium    307
High      185
Name: count, dtype: int64


In [21]:
# Validate observed churn rate by calibrated risk level

risk_validation = (
    research_predictions
    .groupby("CalibratedRiskLevel", observed=False)
    .agg(
        ChurnRate=("ActualChurn", "mean"),
        Churners=("ActualChurn", "sum"),
        Customers=("ActualChurn", "count")
    )
)

risk_validation["ChurnRate"] = (
    risk_validation["ChurnRate"] * 100
)

print("=" * 60)
print("RISK LEVEL VALIDATION")
print("=" * 60)

print(risk_validation.round(2))

RISK LEVEL VALIDATION
                     ChurnRate  Churners  Customers
CalibratedRiskLevel                                
Low                      11.78       108        917
Medium                   44.63       137        307
High                     69.73       129        185


In [22]:
# Calculate the percentage of actual churners captured by each risk level

total_churners = research_predictions["ActualChurn"].sum()

risk_validation["ChurnerCapturePct"] = (
    risk_validation["Churners"] / total_churners * 100
)

print("=" * 60)
print("CHURNER CAPTURE BY RISK LEVEL")
print("=" * 60)

print(
    risk_validation[
        [
            "ChurnRate",
            "Churners",
            "Customers",
            "ChurnerCapturePct"
        ]
    ].round(2)
)

CHURNER CAPTURE BY RISK LEVEL
                     ChurnRate  Churners  Customers  ChurnerCapturePct
CalibratedRiskLevel                                                   
Low                      11.78       108        917              28.88
Medium                   44.63       137        307              36.63
High                     69.73       129        185              34.49


In [23]:
# Calculate cumulative churn capture for retention targeting

risk_order = ["High", "Medium", "Low"]

cumulative_churners = 0

capture_results = []

for risk_level in risk_order:
    row = risk_validation.loc[risk_level]

    cumulative_churners += row["Churners"]

    capture_results.append({
        "TargetedRiskLevels": (
            "High"
            if risk_level == "High"
            else "High + Medium"
            if risk_level == "Medium"
            else "High + Medium + Low"
        ),
        "CustomersTargeted": int(
            risk_validation.loc[
                risk_order[:risk_order.index(risk_level) + 1],
                "Customers"
            ].sum()
        ),
        "ActualChurnersCaptured": int(cumulative_churners),
        "ChurnerCapturePct": (
            cumulative_churners / total_churners * 100
        )
    })

cumulative_capture = pd.DataFrame(capture_results)

print("=" * 60)
print("CUMULATIVE CHURN CAPTURE")
print("=" * 60)

print(cumulative_capture.round(2))

CUMULATIVE CHURN CAPTURE
    TargetedRiskLevels  CustomersTargeted  ActualChurnersCaptured  \
0                 High                185                     129   
1        High + Medium                492                     266   
2  High + Medium + Low               1409                     374   

   ChurnerCapturePct  
0              34.49  
1              71.12  
2             100.00  


In [24]:
# Evaluate retention targeting efficiency

total_customers = len(research_predictions)

cumulative_capture["CustomerPct"] = (
    cumulative_capture["CustomersTargeted"]
    / total_customers * 100
)

cumulative_capture["CaptureEfficiency"] = (
    cumulative_capture["ChurnerCapturePct"]
    / cumulative_capture["CustomerPct"]
)

print("=" * 60)
print("RETENTION TARGETING EFFICIENCY")
print("=" * 60)

print(
    cumulative_capture[
        [
            "TargetedRiskLevels",
            "CustomersTargeted",
            "CustomerPct",
            "ChurnerCapturePct",
            "CaptureEfficiency"
        ]
    ].round(2)
)

RETENTION TARGETING EFFICIENCY
    TargetedRiskLevels  CustomersTargeted  CustomerPct  ChurnerCapturePct  \
0                 High                185        13.13              34.49   
1        High + Medium                492        34.92              71.12   
2  High + Medium + Low               1409       100.00             100.00   

   CaptureEfficiency  
0               2.63  
1               2.04  
2               1.00  


## Research Calibration Conclusion

Probability calibration improved the quality of the research Random Forest probability estimates on the held-out test set.

- Brier Score: **0.1562 → 0.1431**
- Log Loss: **0.4768 → 0.4397**
- ROC-AUC: **0.8208 → 0.8276**

The calibrated probabilities also produced clear risk stratification:

- **Low:** 917 customers, 11.78% observed churn
- **Medium:** 307 customers, 44.63% observed churn
- **High:** 185 customers, 69.73% observed churn

When retention resources are focused on the highest-risk customers, the model captures a concentrated share of observed churners. Targeting the **High + Medium** groups covers 34.92% of the held-out customers while capturing 71.12% of observed churners.

These results support the use of calibrated probability estimates as the foundation for the subsequent revenue-aware retention-prioritization analysis.


In [25]:
research_test_predictions = research_predictions.copy()

research_test_predictions.to_csv(
    "../data/processed/research_test_predictions.csv",
    index=False
)

print("Research test predictions exported successfully.")
print("Shape:", research_test_predictions.shape)
print("File: ../data/processed/research_test_predictions.csv")


Research test predictions exported successfully.
Shape: (1409, 28)
File: ../data/processed/research_test_predictions.csv


### Notebook Handoff

Notebook 04 will consume the exported held-out research predictions from this notebook.

This keeps the workflow sequential:

**Research model → calibration → validated probabilities → Notebook 04 revenue/retention analysis**

No model is retrained unnecessarily in Notebook 04.
